# EXP-1: Fine-tune ConvNeXt-Tiny — LR thấp

Notebook này thực hiện **EXP-1** trong pipeline fine-tuning ConvNeXt-Tiny:

```
BASELINE → EXP-1 (LR thấp) → EXP-2 → EXP-3 → EXP-4 → EXP-5 → EXP-6 → FINAL
```

**Thay đổi duy nhất so với BASELINE:** `LEARNING_RATE` giảm từ `1e-4` xuống `1e-5`.
Tất cả các điều kiện khác (augmentation, loss, optimizer, scheduler, batch size, epochs, data split...) được **giữ nguyên** như file `Comparation_3_model.ipynb`.

**Nguyên tắc pipeline:** Mỗi thực nghiệm mới luôn fine-tune tiếp từ **best checkpoint tốt nhất tính đến thời điểm hiện tại** (dựa trên `val_f1`, tie-break bằng `val_roc_auc`), được theo dõi qua file `best_tracker.json` — **không** nhất thiết là checkpoint của thực nghiệm ngay trước đó. Nếu EXP-1 không cải thiện so với BASELINE, `best_tracker.json` vẫn giữ nguyên BASELINE, và EXP-2 sẽ fine-tune từ BASELINE chứ không phải từ EXP-1.

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm


In [ ]:
import os
import gc
import json
import time
import copy
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

print("All libraries imported successfully.")
print("PyTorch:", torch.__version__)


In [ ]:
# =========================
# Reproducibility
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =========================
# Device
# =========================

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


# =========================
# CUDA optimization
# =========================

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

    # Cho phép TensorFloat-32 trên GPU hỗ trợ
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("CUDA optimization configured.")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
print(os.listdir("/content/drive/MyDrive")[:30])


In [ ]:
# =========================
# Google Drive
# =========================

DRIVE_ROOT = "/content/drive/MyDrive"

ZIP_NAME = "face_frames.zip"

ZIP_PATH = os.path.join(
    DRIVE_ROOT,
    ZIP_NAME
)


# =========================
# Local Colab
# =========================

EXTRACT_DIR = "/content/faceforensics"


# =========================
# Experiment output (dùng LẠI cùng thư mục với notebook so sánh 3 model)
# =========================

EXPERIMENT_DIR = os.path.join(
    DRIVE_ROOT,
    "FaceForensics_Experiments"
)

# Checkpoint/history của BASELINE (đã train ở Comparation_3_model.ipynb)
BASELINE_CHECKPOINT_DIR = os.path.join(
    EXPERIMENT_DIR,
    "checkpoints"
)

BASELINE_CHECKPOINT_PATH = os.path.join(
    BASELINE_CHECKPOINT_DIR,
    "convnext_tiny_best.pth"
)

MODEL_COMPARISON_CSV = os.path.join(
    EXPERIMENT_DIR,
    "results",
    "model_comparison.csv"
)


# =========================
# Thư mục riêng cho pipeline fine-tune ConvNeXt-Tiny
# (EXP-1 -> EXP-6 -> FINAL đều nằm trong đây)
# =========================

FINETUNE_DIR = os.path.join(
    EXPERIMENT_DIR,
    "finetune_convnext_tiny"
)

FT_CHECKPOINT_DIR = os.path.join(
    FINETUNE_DIR,
    "checkpoints"
)

FT_HISTORY_DIR = os.path.join(
    FINETUNE_DIR,
    "histories"
)

FT_RESULT_DIR = os.path.join(
    FINETUNE_DIR,
    "results"
)

# File theo dõi best checkpoint tốt nhất TÍNH ĐẾN THỜI ĐIỂM HIỆN TẠI
# (không phải checkpoint của exp ngay trước đó)
BEST_TRACKER_PATH = os.path.join(
    FINETUNE_DIR,
    "best_tracker.json"
)

# Log tổng hợp tất cả các thực nghiệm fine-tune (append theo từng notebook)
EXPERIMENTS_LOG_PATH = os.path.join(
    FT_RESULT_DIR,
    "experiments_log.csv"
)


for folder in [
    EXPERIMENT_DIR,
    FINETUNE_DIR,
    FT_CHECKPOINT_DIR,
    FT_HISTORY_DIR,
    FT_RESULT_DIR
]:
    os.makedirs(folder, exist_ok=True)


print("ZIP:", ZIP_PATH)
print("Extract:", EXTRACT_DIR)
print("Baseline checkpoint:", BASELINE_CHECKPOINT_PATH)
print("Finetune dir:", FINETUNE_DIR)
print("Best tracker:", BEST_TRACKER_PATH)


In [ ]:
# =========================
# Khởi tạo / đọc best_tracker.json
# =========================
#
# best_tracker.json luôn trỏ tới checkpoint TỐT NHẤT tính đến hiện tại
# (so sánh theo val_f1, tie-break bằng val_roc_auc), bất kể nó đến từ
# BASELINE hay từ bất kỳ EXP nào trước đó. Mỗi notebook EXP-x đọc file
# này để biết nên fine-tune tiếp từ đâu.

def init_best_tracker():
    """Nếu chưa có tracker, khởi tạo từ kết quả BASELINE (convnext_tiny)."""

    if os.path.exists(BEST_TRACKER_PATH):
        with open(BEST_TRACKER_PATH, "r") as f:
            return json.load(f)

    if not os.path.exists(MODEL_COMPARISON_CSV):
        raise FileNotFoundError(
            f"Không tìm thấy {MODEL_COMPARISON_CSV}. "
            "Hãy chạy Comparation_3_model.ipynb trước, "
            "hoặc kiểm tra lại đường dẫn EXPERIMENT_DIR."
        )

    comparison_df = pd.read_csv(MODEL_COMPARISON_CSV)

    baseline_row = comparison_df[
        comparison_df["model"] == "convnext_tiny"
    ].iloc[0]

    tracker = {
        "source_exp": "BASELINE",
        "checkpoint": baseline_row["checkpoint"],
        "best_val_f1": float(baseline_row["best_val_f1"]),
        "best_val_roc_auc": float(baseline_row["best_val_roc_auc"]),
    }

    with open(BEST_TRACKER_PATH, "w") as f:
        json.dump(tracker, f, indent=2)

    return tracker


best_tracker = init_best_tracker()

CURRENT_BEST_CHECKPOINT = best_tracker["checkpoint"]

print("Best checkpoint hiện tại được lấy từ:", best_tracker["source_exp"])
print("Checkpoint path :", CURRENT_BEST_CHECKPOINT)
print("Best val_f1     :", best_tracker["best_val_f1"])
print("Best val_roc_auc:", best_tracker["best_val_roc_auc"])

if not os.path.exists(CURRENT_BEST_CHECKPOINT):
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint: {CURRENT_BEST_CHECKPOINT}\n"
        "Hãy kiểm tra lại model_best.pth đã lưu trên Google Drive."
    )


In [ ]:
if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / 1024**3

    print(
        f"Found ZIP: {ZIP_PATH}"
    )

    print(
        f"ZIP size: {size_gb:.2f} GB"
    )

else:
    raise FileNotFoundError(
        f"Không tìm thấy {ZIP_PATH}\n"
        "Hãy kiểm tra lại tên file hoặc vị trí trên Google Drive."
    )


In [ ]:
if os.path.exists(EXTRACT_DIR):
    print("Extract directory already exists.")
    print("Skip extraction.")

else:

    os.makedirs(EXTRACT_DIR, exist_ok=True)

    print("Extracting dataset...")
    print("This may take some time...")

    start_time = time.time()

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            EXTRACT_DIR
        )

    elapsed = time.time() - start_time

    print(
        f"Extraction completed in "
        f"{elapsed:.1f} seconds."
    )


In [ ]:
print(
    os.listdir(EXTRACT_DIR)
)


In [ ]:
def find_dataset_root(base_dir):

    candidates = []

    for root, dirs, files in os.walk(base_dir):

        dir_names = {
            d.lower()
            for d in dirs
        }

        if (
            "train" in dir_names
            and "val" in dir_names
            and "test" in dir_names
        ):

            candidates.append(root)

    if len(candidates) == 0:

        raise FileNotFoundError(
            "Không tìm thấy thư mục chứa "
            "train/val/test."
        )

    # Chọn candidate đầu tiên
    return candidates[0]


DATA_DIR = find_dataset_root(
    EXTRACT_DIR
)

TRAIN_DIR = os.path.join(
    DATA_DIR,
    "train"
)

VAL_DIR = os.path.join(
    DATA_DIR,
    "val"
)

# TEST chỉ lưu path để giữ cấu trúc.
# KHÔNG đọc test trong quá trình train.
TEST_DIR = os.path.join(
    DATA_DIR,
    "test"
)


print("DATA_DIR : ", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)


In [ ]:
print("\nTrain:")
print(os.listdir(TRAIN_DIR))

print("\nValidation:")
print(os.listdir(VAL_DIR))


In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def count_images(folder):

    count = 0

    for path in Path(folder).rglob("*"):

        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        ):
            count += 1

    return count


def count_class_images(split_dir):

    result = {}

    for class_dir in Path(split_dir).iterdir():

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name.upper()

        result[class_name] = count_images(
            class_dir
        )

    return result


train_counts = count_class_images(
    TRAIN_DIR
)

val_counts = count_class_images(
    VAL_DIR
)


print("========== TRAIN ==========")

for cls, count in train_counts.items():
    print(f"{cls}: {count:,}")


print("\n========== VALIDATION ==========")

for cls, count in val_counts.items():
    print(f"{cls}: {count:,}")


print("\nTotal TRAIN:")
print(
    f"{sum(train_counts.values()):,}"
)

print("\nTotal VALIDATION:")
print(
    f"{sum(val_counts.values()):,}"
)


In [ ]:
# =========================
# Image
# =========================

IMG_SIZE = 224


# =========================
# Training
# =========================
#
# EXP-1: CHỈ thay đổi LEARNING_RATE (1e-4 -> 1e-5).
# Toàn bộ các tham số khác giữ NGUYÊN như BASELINE
# (Comparation_3_model.ipynb).

BATCH_SIZE = 64
EPOCHS = 10

LEARNING_RATE = 1e-5   # <-- THAY ĐỔI DUY NHẤT so với baseline (1e-4)
WEIGHT_DECAY = 1e-4


# =========================
# DataLoader
# =========================

NUM_WORKERS = 4

PIN_MEMORY = (
    device.type == "cuda"
)

PERSISTENT_WORKERS = (
    NUM_WORKERS > 0
)

PREFETCH_FACTOR = 2


# =========================
# Mixed Precision
# =========================

USE_AMP = (
    device.type == "cuda"
)


# =========================
# Classes
# =========================

# FAKE là positive class
CLASS_TO_LABEL = {
    "REAL": 0,
    "FAKE": 1
}

LABEL_TO_CLASS = {
    0: "REAL",
    1: "FAKE"
}


# =========================
# Experiment identity
# =========================

MODEL_NAME = "convnext_tiny"
EXP_NAME = "exp1_lr_low"
EXP_DESCRIPTION = "Fine-tune LR thap (1e-5), giu nguyen cac dieu kien khac so voi baseline"


print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("LR:", LEARNING_RATE)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("AMP:", USE_AMP)
print("EXP_NAME:", EXP_NAME)


In [ ]:
train_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    # Horizontal flip
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    # Rotation nhẹ
    transforms.RandomRotation(
        degrees=10
    ),

    # Thay đổi ánh sáng/màu sắc nhẹ
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


print("Transforms ready (unchanged from baseline).")


In [ ]:
def create_dataframe(
    split_dir
):

    records = []

    split_path = Path(
        split_dir
    )

    for class_dir in split_path.iterdir():

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name.upper()

        if class_name not in CLASS_TO_LABEL:
            print(
                f"Warning: ignored class "
                f"{class_name}"
            )
            continue

        label = CLASS_TO_LABEL[
            class_name
        ]

        for image_path in class_dir.rglob("*"):

            if (
                image_path.is_file()
                and image_path.suffix.lower()
                in IMAGE_EXTENSIONS
            ):

                records.append({

                    "image_path":
                        str(image_path),

                    "label":
                        label,

                    "class_name":
                        class_name

                })

    return pd.DataFrame(
        records
    )


train_df = create_dataframe(
    TRAIN_DIR
)

val_df = create_dataframe(
    VAL_DIR
)


print(
    "TRAIN:",
    f"{len(train_df):,}"
)

print(
    "VAL:",
    f"{len(val_df):,}"
)


In [ ]:
print("\nTRAIN")
print(
    train_df["class_name"].value_counts()
)

print("\nVAL")
print(
    val_df["class_name"].value_counts()
)


In [ ]:
class FaceForensicsDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        image_path = row[
            "image_path"
        ]

        label = int(
            row["label"]
        )


        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            print(
                f"\nError loading:"
                f"\n{image_path}"
            )

            raise e


        if self.transform is not None:

            image = self.transform(
                image
            )


        return image, label


In [ ]:
train_dataset = FaceForensicsDataset(
    train_df,
    transform=train_transform
)

val_dataset = FaceForensicsDataset(
    val_df,
    transform=val_transform
)


train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    persistent_workers=PERSISTENT_WORKERS,

    prefetch_factor=PREFETCH_FACTOR
        if NUM_WORKERS > 0
        else None
)


val_loader = DataLoader(
    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    persistent_workers=PERSISTENT_WORKERS,

    prefetch_factor=PREFETCH_FACTOR
        if NUM_WORKERS > 0
        else None
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


In [ ]:
def create_model(
    model_name
):

    if model_name == "convnext_tiny":

        model = convnext_tiny(
            weights=ConvNeXt_Tiny_Weights.DEFAULT
        )

        in_features = (
            model.classifier[-1]
            .in_features
        )

        model.classifier[-1] = nn.Linear(
            in_features,
            2
        )

    else:

        raise ValueError(
            f"Unknown model: "
            f"{model_name}"
        )

    return model


def load_finetune_weights(
    model,
    checkpoint_path
):
    """Load trọng số từ best checkpoint trước đó (BASELINE hoặc EXP trước)."""

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False  # checkpoint tu train, tin tuong duoc (PyTorch 2.6+ mac dinh weights_only=True)
    )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint
    )

    model.load_state_dict(state_dict)

    print(
        f"Loaded weights from: {checkpoint_path}"
    )

    if "metrics" in checkpoint:
        print("  Source metrics:", checkpoint["metrics"])

    return model


In [ ]:
model = create_model(MODEL_NAME)

model = load_finetune_weights(
    model,
    CURRENT_BEST_CHECKPOINT
)

model = model.to(device)

if device.type == "cuda":
    model = model.to(
        memory_format=torch.channels_last
    )


total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"{MODEL_NAME:20s}"
    f" | Params: "
    f"{total_params / 1e6:.2f}M"
    f" | Trainable: "
    f"{trainable_params / 1e6:.2f}M"
)


In [ ]:
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)


print("Class counts:")

for label, count in class_counts.items():

    print(
        LABEL_TO_CLASS[label],
        ":",
        f"{count:,}"
    )


total_samples = len(
    train_df
)

num_classes = 2


class_weights = []

for label in range(num_classes):

    count = class_counts[
        label
    ]

    weight = (
        total_samples
        / (num_classes * count)
    )

    class_weights.append(
        weight
    )


class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)


print("\nClass weights:")

for label, weight in enumerate(
    class_weights
):

    print(
        LABEL_TO_CLASS[label],
        ":",
        f"{weight.item():.4f}"
    )


In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print(criterion)


In [ ]:
def create_optimizer(
    model
):

    return torch.optim.AdamW(
        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


In [ ]:
def create_scheduler(
    optimizer
):

    return torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=1,

        min_lr=1e-6
    )


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []


    progress = tqdm(
        loader,
        desc="Train",
        leave=False
    )


    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        # channels_last tối ưu CNN trên CUDA
        if device.type == "cuda":

            images = images.contiguous(
                memory_format=torch.channels_last
            )


        optimizer.zero_grad(
            set_to_none=True
        )


        # Mixed Precision
        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP
        ):

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )


        if USE_AMP:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        running_loss += (
            loss.item()
            * images.size(0)
        )


        preds = torch.argmax(
            outputs,
            dim=1
        )


        all_preds.extend(
            preds.detach()
            .cpu()
            .numpy()
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )


    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )


    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        pos_label=1,
        zero_division=0
    )


    return epoch_loss, epoch_f1


In [ ]:
def validate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            if device.type == "cuda":

                images = images.contiguous(
                    memory_format=torch.channels_last
                )


            with torch.amp.autocast(
                device_type=device.type,
                enabled=USE_AMP
            ):

                outputs = model(
                    images
                )

                loss = criterion(
                    outputs,
                    labels
                )


            running_loss += (
                loss.item()
                * images.size(0)
            )


            probabilities = torch.softmax(
                outputs,
                dim=1
            )


            preds = torch.argmax(
                outputs,
                dim=1
            )


            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_probs.extend(
                probabilities[:, 1]
                .cpu()
                .numpy()
            )


    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )


    accuracy = accuracy_score(
        all_labels,
        all_preds
    )


    precision = precision_score(
        all_labels,
        all_preds,
        pos_label=1,
        zero_division=0
    )


    recall = recall_score(
        all_labels,
        all_preds,
        pos_label=1,
        zero_division=0
    )


    f1 = f1_score(
        all_labels,
        all_preds,
        pos_label=1,
        zero_division=0
    )


    auc = roc_auc_score(
        all_labels,
        all_probs
    )


    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc
    }


In [ ]:
def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    metrics,
    best_f1
):

    checkpoint_path = os.path.join(
        FT_CHECKPOINT_DIR,
        f"{MODEL_NAME}_{EXP_NAME}_best.pth"
    )


    checkpoint = {

        "model_name":
            MODEL_NAME,

        "exp_name":
            EXP_NAME,

        "finetuned_from":
            CURRENT_BEST_CHECKPOINT,

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict()
            if USE_AMP
            else None,

        "best_f1":
            best_f1,

        "metrics":
            metrics,

        "config": {

            "img_size":
                IMG_SIZE,

            "batch_size":
                BATCH_SIZE,

            "epochs":
                EPOCHS,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "seed":
                SEED

        }
    }


    torch.save(
        checkpoint,
        checkpoint_path
    )


    return checkpoint_path


In [ ]:
def train_model():

    print("\n")
    print("=" * 70)
    print(
        f"START FINE-TUNE: "
        f"{MODEL_NAME} | {EXP_NAME}"
    )
    print(
        f"Description: {EXP_DESCRIPTION}"
    )
    print(
        f"Starting from: {CURRENT_BEST_CHECKPOINT}"
    )
    print("=" * 70)


    # =========================
    # Optimizer / Scheduler / AMP
    # (model đã được tạo + load checkpoint + move to device ở cell trước)
    # =========================

    optimizer = create_optimizer(
        model
    )

    scheduler = create_scheduler(
        optimizer
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )


    # =========================
    # History
    # =========================

    history = []

    best_f1 = -1.0
    best_auc = -1.0
    best_epoch = -1


    # =========================
    # Training
    # =========================

    total_start = time.time()


    for epoch in range(
        EPOCHS
    ):

        epoch_start = time.time()


        print(
            f"\nEpoch "
            f"{epoch + 1}/{EPOCHS}"
        )


        # ---------------------
        # TRAIN
        # ---------------------

        train_loss, train_f1 = train_one_epoch(

            model,

            train_loader,

            criterion,

            optimizer,

            scaler
        )


        # ---------------------
        # VALIDATION
        # ---------------------

        val_metrics = validate(

            model,

            val_loader,

            criterion
        )


        # ---------------------
        # Scheduler
        # ---------------------

        scheduler.step(
            val_metrics["f1"]
        )


        current_lr = (
            optimizer.param_groups[0]["lr"]
        )


        epoch_time = (
            time.time()
            - epoch_start
        )


        # ---------------------
        # Save history
        # ---------------------

        row = {

            "epoch":
                epoch + 1,

            "train_loss":
                train_loss,

            "train_f1":
                train_f1,

            "val_loss":
                val_metrics["loss"],

            "val_accuracy":
                val_metrics["accuracy"],

            "val_precision":
                val_metrics["precision"],

            "val_recall":
                val_metrics["recall"],

            "val_f1":
                val_metrics["f1"],

            "val_roc_auc":
                val_metrics["roc_auc"],

            "learning_rate":
                current_lr,

            "epoch_time_sec":
                epoch_time
        }


        history.append(
            row
        )


        # ---------------------
        # Print
        # ---------------------

        print(
            f"Train Loss : "
            f"{train_loss:.4f}"
        )

        print(
            f"Train F1   : "
            f"{train_f1:.4f}"
        )

        print(
            f"Val Loss   : "
            f"{val_metrics['loss']:.4f}"
        )

        print(
            f"Val F1     : "
            f"{val_metrics['f1']:.4f}"
        )

        print(
            f"Val ROC-AUC: "
            f"{val_metrics['roc_auc']:.4f}"
        )

        print(
            f"LR         : "
            f"{current_lr:.2e}"
        )

        print(
            f"Epoch time : "
            f"{epoch_time:.1f}s"
        )


        # ---------------------
        # BEST CHECKPOINT (trong nội bộ EXP-1)
        # ---------------------

        is_better = (

            val_metrics["f1"]
            > best_f1

            or

            (
                val_metrics["f1"]
                == best_f1

                and

                val_metrics["roc_auc"]
                > best_auc
            )
        )


        if is_better:

            best_f1 = (
                val_metrics["f1"]
            )

            best_auc = (
                val_metrics["roc_auc"]
            )

            best_epoch = (
                epoch + 1
            )


            checkpoint_path = save_checkpoint(

                model,

                optimizer,

                scheduler,

                scaler,

                epoch + 1,

                val_metrics,

                best_f1
            )


            print(
                f"✓ BEST MODEL (EXP-1) SAVED"
            )

            print(
                f"  F1: "
                f"{best_f1:.4f}"
            )

            print(
                f"  AUC: "
                f"{best_auc:.4f}"
            )

            print(
                f"  Path: "
                f"{checkpoint_path}"
            )


    total_time = (
        time.time()
        - total_start
    )


    # =========================
    # Save history
    # =========================

    history_df = pd.DataFrame(
        history
    )

    history_path = os.path.join(

        FT_HISTORY_DIR,

        f"{MODEL_NAME}_{EXP_NAME}_history.csv"
    )

    history_df.to_csv(
        history_path,
        index=False
    )


    # =========================
    # Summary
    # =========================

    summary = {

        "exp_name":
            EXP_NAME,

        "description":
            EXP_DESCRIPTION,

        "model":
            MODEL_NAME,

        "finetuned_from":
            CURRENT_BEST_CHECKPOINT,

        "learning_rate":
            LEARNING_RATE,

        "best_epoch":
            best_epoch,

        "best_val_f1":
            best_f1,

        "best_val_roc_auc":
            best_auc,

        "total_training_time_sec":
            total_time,

        "average_epoch_time_sec":
            history_df[
                "epoch_time_sec"
            ].mean(),

        "checkpoint":
            os.path.join(
                FT_CHECKPOINT_DIR,
                f"{MODEL_NAME}_{EXP_NAME}_best.pth"
            ),

        "history":
            history_path
    }


    print("\n")
    print("=" * 70)
    print(
        f"FINISHED: {EXP_NAME}"
    )
    print("=" * 70)

    print(
        f"Best Epoch : {best_epoch}"
    )

    print(
        f"Best F1    : {best_f1:.4f}"
    )

    print(
        f"Best AUC   : {best_auc:.4f}"
    )

    print(
        f"Avg epoch  : "
        f"{summary['average_epoch_time_sec']:.1f}s"
    )

    print(
        f"Total time : "
        f"{total_time / 60:.2f} min"
    )


    return summary, history_df


In [ ]:
exp1_summary, exp1_history = train_model()


In [ ]:
# =========================
# So sánh EXP-1 với best hiện tại, cập nhật best_tracker.json NẾU tốt hơn
# =========================

is_new_best = (

    exp1_summary["best_val_f1"]
    > best_tracker["best_val_f1"]

    or

    (
        exp1_summary["best_val_f1"]
        == best_tracker["best_val_f1"]

        and

        exp1_summary["best_val_roc_auc"]
        > best_tracker["best_val_roc_auc"]
    )
)

print(f"Previous best ({best_tracker['source_exp']}):")
print(f"  val_f1     = {best_tracker['best_val_f1']:.4f}")
print(f"  val_roc_auc= {best_tracker['best_val_roc_auc']:.4f}")

print(f"\nEXP-1 result:")
print(f"  val_f1     = {exp1_summary['best_val_f1']:.4f}")
print(f"  val_roc_auc= {exp1_summary['best_val_roc_auc']:.4f}")

if is_new_best:

    best_tracker = {
        "source_exp": EXP_NAME,
        "checkpoint": exp1_summary["checkpoint"],
        "best_val_f1": exp1_summary["best_val_f1"],
        "best_val_roc_auc": exp1_summary["best_val_roc_auc"],
    }

    with open(BEST_TRACKER_PATH, "w") as f:
        json.dump(best_tracker, f, indent=2)

    print(
        f"\n✓ EXP-1 CẢI THIỆN kết quả. "
        f"best_tracker.json đã được cập nhật -> nguồn cho EXP-2 sẽ là EXP-1."
    )

else:

    print(
        f"\n✗ EXP-1 KHÔNG cải thiện so với best hiện tại "
        f"({best_tracker['source_exp']}). "
        f"best_tracker.json GIỮ NGUYÊN -> EXP-2 vẫn fine-tune từ "
        f"{best_tracker['source_exp']}, không phải từ EXP-1."
    )


In [ ]:
# =========================
# Ghi log vào experiments_log.csv (tổng hợp toàn bộ pipeline EXP-1..FINAL)
# =========================

log_row = pd.DataFrame([{
    "exp_name": EXP_NAME,
    "description": EXP_DESCRIPTION,
    "model": MODEL_NAME,
    "finetuned_from": CURRENT_BEST_CHECKPOINT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "best_epoch": exp1_summary["best_epoch"],
    "best_val_f1": exp1_summary["best_val_f1"],
    "best_val_roc_auc": exp1_summary["best_val_roc_auc"],
    "total_training_time_sec": exp1_summary["total_training_time_sec"],
    "checkpoint": exp1_summary["checkpoint"],
    "is_new_best": is_new_best
}])

if os.path.exists(EXPERIMENTS_LOG_PATH):
    existing_log = pd.read_csv(EXPERIMENTS_LOG_PATH)
    # Ghi đè nếu exp_name đã tồn tại (chạy lại cùng notebook)
    existing_log = existing_log[
        existing_log["exp_name"] != EXP_NAME
    ]
    full_log = pd.concat(
        [existing_log, log_row],
        ignore_index=True
    )
else:
    full_log = log_row

full_log.to_csv(
    EXPERIMENTS_LOG_PATH,
    index=False
)

print("Saved:", EXPERIMENTS_LOG_PATH)
display(full_log)


In [ ]:
# =========================
# So sánh trực quan: BASELINE vs EXP-1
# =========================

baseline_history_path = os.path.join(
    EXPERIMENT_DIR,
    "histories",
    "convnext_tiny_history.csv"
)

baseline_history = pd.read_csv(baseline_history_path)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = [
    ("val_f1", "Validation F1"),
    ("val_roc_auc", "Validation ROC-AUC"),
    ("val_loss", "Validation Loss"),
]

for ax, (col, title) in zip(axes, metrics_to_plot):

    ax.plot(
        baseline_history["epoch"],
        baseline_history[col],
        marker="o",
        label="BASELINE"
    )

    ax.plot(
        exp1_history["epoch"],
        exp1_history[col],
        marker="o",
        label="EXP-1 (LR=1e-5)"
    )

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(col)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
print("EXP-1 hoàn tất.")
print()
print("Checkpoint của EXP-1     :", exp1_summary["checkpoint"])
print("Checkpoint TỐT NHẤT hiện tại (dùng cho EXP-2):", best_tracker["checkpoint"])
print()
print("-> Notebook EXP-2 chỉ cần đọc lại best_tracker.json (cell 'Best tracker' ở trên)")
print("   để tự động fine-tune tiếp từ checkpoint tốt nhất tính đến thời điểm này.")
